# GuideAI model v3 — training on Google Colab

Trains YOLO11n on the 45-class GuideAI dataset (the 25 existing classes + 20 everyday indoor classes) and exports the phone model.

**Before you start**
1. In Google Drive, make a folder named **`GuideAI`** in *My Drive*.
2. Upload these two files from `BUTA APP/model/colab/` into it:
   - `guideai_v3_dataset.zip` (the dataset)
   - `guideai_v2_best.pt` (the current model, used as the starting point)
3. In Colab: **Runtime → Change runtime type → T4 GPU**.
4. Run the cells in order (click the ▶ on each, or Runtime → Run all).

Progress is saved to Drive after every epoch, so if Colab disconnects, run cells 1–4, then the **Resume** cell.

In [ ]:
# 1. Check that a GPU is attached (should show a Tesla T4 or better)
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# 2. Connect Google Drive (a window asks for permission)
from google.colab import drive
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/GuideAI'
import os
for f in ('guideai_v3_dataset.zip', 'guideai_v2_best.pt'):
    print(f, 'found' if os.path.exists(f'{DRIVE}/{f}') else 'MISSING - upload it to My Drive/GuideAI')

In [ ]:
# 3. Copy the dataset to the fast local disk and unzip it (a few minutes)
!cp "$DRIVE/guideai_v3_dataset.zip" /content/
!unzip -q -o /content/guideai_v3_dataset.zip -d /content/
!rm /content/guideai_v3_dataset.zip
!head -8 /content/merged_v3/data.yaml
!echo train images: $(ls /content/merged_v3/images/train | wc -l)

In [ ]:
# 4. Install the same Ultralytics version used on the PC
!pip install -q ultralytics==8.4.174 || pip install -q ultralytics
import ultralytics; ultralytics.checks()

In [ ]:
# 5. Train. Starts from the current GuideAI model (v2): its knowledge of the 25 classes is kept,
#    and the output layer is rebuilt for 45 classes. About 2-4 hours on a T4.
from ultralytics import YOLO
model = YOLO(f'{DRIVE}/guideai_v2_best.pt')
model.train(
    data='/content/merged_v3/data.yaml',
    imgsz=320,            # the app's input size; do not change
    epochs=60,
    patience=15,          # stops early if it stops improving
    batch=64,
    workers=4,
    project=f'{DRIVE}/runs',   # saved to Drive after every epoch
    name='guideai_v3',
    exist_ok=True,
    plots=True,
)

In [ ]:
# Resume — ONLY if Colab disconnected during training (run cells 1-4 first)
from ultralytics import YOLO
YOLO(f'{DRIVE}/runs/guideai_v3/weights/last.pt').train(resume=True)

In [ ]:
# 6. Test on the held-out test photos (never used in training): overall and per class
from ultralytics import YOLO
best = YOLO(f'{DRIVE}/runs/guideai_v3/weights/best.pt')
r = best.val(data='/content/merged_v3/data.yaml', split='test', imgsz=320, plots=False)
print(f'\nOVERALL  precision {r.box.mp:.3f}  recall {r.box.mr:.3f}  mAP50 {r.box.map50:.3f}  mAP50-95 {r.box.map:.3f}\n')
for i, c in enumerate(r.box.ap_class_index):
    p, rc, ap50, ap = r.box.class_result(i)
    print(f'{r.names[int(c)]:15s} P {p:.3f}  R {rc:.3f}  mAP50 {ap50:.3f}')

In [ ]:
# 7. Export the phone model (TFLite, 320 px) and save it to Drive
import glob, shutil
out = best.export(format='tflite', imgsz=320)
print('exported:', out)
for f in glob.glob(f'{DRIVE}/runs/guideai_v3/weights/best_saved_model/*float32.tflite'):
    shutil.copy(f, f'{DRIVE}/guideai_v3_float32.tflite')
    print('copied to Drive:', f'{DRIVE}/guideai_v3_float32.tflite')

## When it's done

From **My Drive/GuideAI**, download:
- `runs/guideai_v3/weights/best.pt`
- `guideai_v3_float32.tflite`
- the whole `runs/guideai_v3` folder if you want the training charts

Put them in `BUTA APP/model/runs/guideai_v3/` on the PC and tell Claude. It will then check the model against the current one on the same test photos (including false "stairs"), add the 45-class label list to the app, and only then put it on the phone.